# 04e. Veto de mãe

Censo sem CPF no [`04_atribuir.ipynb`](04_atribuir.ipynb) cujo **melhor par**
cai no veto de mãe: as duas mães preenchidas, nem iguais nem prefixo
(`n ≥ 2` no começo), e não é o caso de nome completo e data iguais.

O [`04c_decompor_sem_cpf.ipynb`](04c_decompor_sem_cpf.ipynb) mistura isso com
primeiro nome ausente ou com mais de 1 de Levenshtein. Aqui os dois ficam
separados. A escada já aceita Jaro-Winkler da mãe ≥ 0,75, e de 0,95 a 0,99
não aplica esses vetos. Este notebook continua mostrando o corte estrito.


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

('censo_limpo_aplicacao', 'cpf_limpo_aplicacao')

In [2]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

print(
    'Pares no parquet:',
    f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}",
)
print(
    'Atribuições:',
    f"{con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]:,}",
)


Pares no parquet: 6,598,211
Atribuições: 3,905,732


Melhor par de quem não saiu no 04. `veto_mae` exige primeiro
nome preenchido e com no máximo 1 de Levenshtein: se o primeiro nome já
reprova, o par vai para `so_primeiro` e não entra na leitura da mãe.


In [3]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_mae = f"least(len({tok_c}), len({tok_p}))"
prefixo_mae = (
    f"{n_mae} >= 2 AND "
    f"list_slice({tok_c}, 1, {n_mae}) = list_slice({tok_p}, 1, {n_mae})"
)
# Nome completo e data iguais passam mesmo com mãe diferente.
excecao = (
    "ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL "
    "AND ca.nome_completo_phon = pb.nome_completo_phon "
    "AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
      SELECT 1 FROM atribuicao_2 a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute('''
CREATE OR REPLACE TABLE melhor_par AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_sem AS
SELECT
    s.unique_id AS unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN 'sem_par'
        WHEN ca.primeiro_nome_phon IS NULL
             OR pb.primeiro_nome_phon IS NULL
             OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) > 1
            THEN 'so_primeiro'
        WHEN ca.nome_mae_phon IS NOT NULL
             AND pb.nome_mae_phon IS NOT NULL
             AND ca.nome_mae_phon <> pb.nome_mae_phon
             AND NOT ({prefixo_mae})
             AND NOT ({excecao})
            THEN 'veto_mae'
        ELSE 'passaria'
    END AS porta
FROM censo_sem_cpf s
LEFT JOIN melhor_par p ON p.unique_id_censo = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
''')

display(con.execute('''
SELECT
    porta,
    COUNT(*) AS n,
    SUM(CASE WHEN match_probability >= 0.75 THEN 1 ELSE 0 END) AS n_p075,
    ROUND(AVG(match_probability), 3) AS avg_p
FROM melhor_sem
GROUP BY 1
ORDER BY n DESC
''').df())


,porta,n,n_p075,avg_p
0,sem_par,1463222,0.0,NaN
1,passaria,555417,251187.0,0.556
2,veto_mae,77789,18045.0,0.369
3,so_primeiro,76870,64547.0,0.840


No veto de mãe, como os dois nomes da mãe se afastam, ao lado
do nome da pessoa e da data. `mesmas_palavras_ordem` e `contem_sem_prefixo`
são o lugar em que a comparação pode estar rígida demais: as palavras batem,
a regra de prefixo não.


In [4]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_c = f"len({tok_c})"
n_p = f"len({tok_p})"
inter = f"len(list_intersect({tok_c}, {tok_p}))"

con.execute(f'''
CREATE OR REPLACE TABLE veto_mae AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.nome_mae_phon AS mae_censo,
    pb.nome_mae_phon AS mae_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) AS jw_mae,
    CASE
        WHEN list_sort({tok_c}) = list_sort({tok_p}) AND {tok_c} <> {tok_p}
            THEN 'mesmas_palavras_ordem'
        WHEN {tok_c} = {tok_p}
            THEN 'mesmas_palavras'
        WHEN {inter} = least({n_c}, {n_p}) AND {n_c} <> {n_p}
            THEN 'contem_sem_prefixo'
        WHEN jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) >= 0.92
            THEN 'parecidas'
        WHEN {inter} >= 1
            THEN 'alguma_palavra'
        ELSE 'sem_palavra'
    END AS mae_dif,
    CASE
        WHEN ca.nome_completo_phon = pb.nome_completo_phon THEN 'exato'
        WHEN ca.primeiro_nome_phon = pb.primeiro_nome_phon
             AND ca.ultimo_nome_phon = pb.ultimo_nome_phon THEN 'pontas'
        ELSE 'outro'
    END AS nome_pessoa,
    CASE
        WHEN ca.data_nascimento IS NULL AND pb.data_nascimento IS NULL THEN 'as_duas_nulas'
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'uma_nula'
        WHEN ca.data_nascimento = pb.data_nascimento THEN 'exata'
        ELSE 'diferente'
    END AS data_par
FROM melhor_sem m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE m.porta = 'veto_mae'
''')

n_excecao = con.execute('''
SELECT COUNT(*) FROM veto_mae
WHERE nome_pessoa = 'exato' AND data_par = 'exata'
''').fetchone()[0]
print('Veto de mãe:', f"{con.execute('SELECT COUNT(*) FROM veto_mae').fetchone()[0]:,}")
print('Nome e data exatos dentro do veto (tem de ser 0):', n_excecao)

display(con.execute('''
SELECT
    mae_dif,
    nome_pessoa,
    data_par,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(jw_mae), 3) AS avg_jw_mae
FROM veto_mae
GROUP BY 1, 2, 3
ORDER BY n DESC
''').df())


Veto de mãe: 77,789
Nome e data exatos dentro do veto (tem de ser 0): 0


,mae_dif,nome_pessoa,data_par,n,pct,avg_p,avg_jw_mae
0,alguma_palavra,exato,diferente,27387,35.2,0.314,0.678
1,alguma_palavra,exato,uma_nula,14231,18.3,0.320,0.675
2,sem_palavra,exato,diferente,9179,11.8,0.389,0.594
3,sem_palavra,exato,uma_nula,6132,7.9,0.396,0.589
4,alguma_palavra,outro,exata,3100,4.0,0.815,0.666
5,sem_palavra,outro,exata,2789,3.6,0.812,0.593
6,parecidas,exato,diferente,1890,2.4,0.078,0.961
7,alguma_palavra,pontas,exata,1800,2.3,0.746,0.666
8,sem_palavra,pontas,exata,1723,2.2,0.734,0.601
9,alguma_palavra,pontas,diferente,1682,2.2,0.161,0.692


Amostra. Primeiro as mães com as mesmas palavras ou uma dentro
da outra. Depois as parecidas com nome da pessoa exato. Por último, mãe sem
palavra em comum e nome da pessoa exato: o veto no caso claro.


In [5]:
cols = '''
    match_probability,
    mae_dif,
    nome_pessoa,
    data_par,
    ROUND(jw_mae, 3) AS jw_mae,
    nome_censo,
    nome_cpf,
    mae_censo,
    mae_cpf,
    dob_censo,
    dob_cpf
'''

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif IN ('mesmas_palavras_ordem', 'mesmas_palavras', 'contem_sem_prefixo')
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'parecidas' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'sem_palavra' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 20
''').df())


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.949632,contem_sem_prefixo,exato,diferente,0.727,ANTONIU JOZI KONSEIKAU SIUVA NETU,ANTONIU JOZI KONSEIKAU SIUVA NETU,SANTANA SILVA,ANA MARIA SANTANA SILVA,2004-03-01,2004-02-01
1,0.949632,contem_sem_prefixo,exato,diferente,0.680,LEANDRU MARTINS ROXA,LEANDRU MARTINS ROXA,IRENE ROCHA,IRENE NASCISA MARTINS ROCHA,1984-11-01,1984-09-01
2,0.949632,contem_sem_prefixo,exato,diferente,0.699,JOZI ENRIKI ROMA KOSTA,JOZI ENRIKI ROMA KOSTA,NAZARE RODRIGUES,VERONICA NAZARE RODRIGUES ROMA,2013-09-15,2013-11-15
3,0.949632,contem_sem_prefixo,exato,diferente,0.744,JOZI RODOUFU MEDEIRUS SANTUS,JOZI RODOUFU MEDEIRUS SANTUS,RAIMUNDA MEDEIROS,MARIA RAIMUNDA MEDEIROS SILVA,2015-02-14,2015-12-14
4,0.949632,contem_sem_prefixo,exato,diferente,0.690,JONATAN FRAZAU FARIAS,JONATAN FRAZAU FARIAS,APARECIDA FRAZAO,MARIA APARECIDA FRAZAO SANTOS FARIAS,2020-09-08,2020-11-08
5,0.949632,contem_sem_prefixo,exato,diferente,0.624,AZAFI BARBOZA BARUS,AZAFI BARBOZA BARUS,DEBORA BARROS,ANTONIA DEBORA PROFESSOR BARBOSA BARROS,2014-09-06,2014-11-06
6,0.949632,contem_sem_prefixo,exato,diferente,0.709,VAUTER FRANSISKU AUMEIDA JUNIOR,VAUTER FRANSISKU AUMEIDA JUNIOR,SOCORRO SANTOS,MARIA SOCORRO SANTOS GOMES ALMEIDA,1996-07-07,1996-02-07
7,0.949632,contem_sem_prefixo,exato,diferente,0.681,ROSMARIU SIUVA LEAU,ROSMARIU SIUVA LEAU,ALZIRA LEAL,MARIA ALZIRA SILVA LEAL,1962-05-01,1962-03-01
8,0.949632,contem_sem_prefixo,exato,diferente,0.586,ANA REBEKA PAXEKU VIEIRA,ANA REBEKA PAXEKU VIEIRA,NONATA VIEIRA,RAIMUNDA NONATA PACHECO VIEIRA,2001-06-20,2001-07-20
9,0.949632,contem_sem_prefixo,exato,diferente,0.702,KLARISI LIMA DAMASENA,KLARISI LIMA DAMASENA,SILVANETE FRANCA LIMA,MARIA SILVANETE FRANCA LIMA DAMACENA,2014-06-10,2014-04-10


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.697117,parecidas,exato,uma_nula,0.983,MARIA FATIMA MARKIS OLIVEIRA,MARIA FATIMA MARKIS OLIVEIRA,ELENA SOUSA MARQUES,HELENA SOUSA MARQUES,None,2014-08-27
1,0.697117,parecidas,exato,uma_nula,0.978,MANUELI KONSEIKAU SOUZA,MANUELI KONSEIKAU SOUZA,JHESCYKA CONCEICAO,JHESCIKA CONCEICAO,None,2016-08-27
2,0.697117,parecidas,exato,uma_nula,0.959,LAIS KONSEIKAU SOUZA,LAIS KONSEIKAU SOUZA,MARIA FRANCISA CONCEICAO,MARIA FRANCISCA CONCEICAO,None,2004-02-16
3,0.697117,parecidas,exato,uma_nula,0.981,LUIS GUSTAVU PIRIS SANTUS,LUIS GUSTAVU PIRIS SANTUS,LUCIMARA PIRIS SANTOS,LUCIMARA PIRES SANTOS,None,2013-02-03
4,0.697117,parecidas,exato,uma_nula,0.988,IARA AUMEIDA SIUVA,IARA AUMEIDA SIUVA,ASUNCAO ALMEIDA SILVA,ASSUNCAO ALMEIDA SILVA,None,1995-08-23
5,0.697117,parecidas,exato,uma_nula,0.921,ADEMAR OLIVEIRA KARVALU,ADEMAR OLIVEIRA KARVALU,LEINICI OLIVEIRA CARVALHO,LENICE OLIVEIRA CARVALHO,None,1995-02-20
6,0.697117,parecidas,exato,uma_nula,0.988,ENZU GABRIEU KANTANHEDI,ENZU GABRIEU KANTANHEDI,LETICIA CANTANHEDE,LETRICIA CANTANHEDE,None,2020-04-09
7,0.697117,parecidas,exato,uma_nula,0.981,ROMULU SANTUS KRUS,ROMULU SANTUS KRUS,FRANCISCA SABTOS CRUZ,FRANCISCA SANTOS CRUZ,None,1989-12-09
8,0.668204,parecidas,exato,uma_nula,0.927,MARIA SOKORU LOPIS LIMA,MARIA SOKORU LOPIS LIMA,MARIA LOPES LIMA,RAIMUNDA LOPES LIMA,None,1971-05-24
9,0.657151,parecidas,exato,uma_nula,0.931,JEFERSON MARTINS DIAS,JEFERSON MARTINS DIAS,RITA CASSIO MARTINS DIAS,RITA CASSIA MARTINS DIAS,None,2002-08-03


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.984569,sem_palavra,exato,diferente,0.532,ISTEFANI BLENDA PEREIRA AMORIN,ISTEFANI BLENDA PEREIRA AMORIN,LUCIBELA ALVES JESUS,MARIA DAUVERCI PEREIRA,2008-08-08,2000-08-08
1,0.984569,sem_palavra,exato,diferente,0.497,TALITA EMILI OLIVEIRA REGU,TALITA EMILI OLIVEIRA REGU,EDNA SOUSA REGO,VERA ALICE OLIVEIRA DIAS,2013-11-13,2003-11-13
2,0.984569,sem_palavra,exato,diferente,0.696,NAIRA MONIKI PEREIRA SOUZA,NAIRA MONIKI PEREIRA SOUZA,MARIANA FERREIRA,FRANCINILDE SIQUEIRA PEREIRA,2015-04-14,2012-04-14
3,0.984569,sem_palavra,exato,diferente,0.542,DANILU OLIVEIRA MOURA,DANILU OLIVEIRA MOURA,NATALIA SANTOS OLIVEIRA,RAIMUNDA FERREIRA MOURA,2003-08-02,2005-08-02
4,0.984569,sem_palavra,exato,diferente,0.603,IANA GABRIELI RODRIGUIS OLIVEIRA,IANA GABRIELI RODRIGUIS OLIVEIRA,WELITA OLIVEIRA SOUSA,MARIA RODRIGUES GOMES,2013-10-20,2015-10-20
5,0.984569,sem_palavra,exato,diferente,0.669,RAMONA PAIVA SANTUS,RAMONA PAIVA SANTUS,LUCIANA RODRIGUES SILVA,CLEUDIMAR CONCEICAO PAIVA,2006-06-19,2008-06-19
6,0.984569,sem_palavra,exato,diferente,0.662,UGU FELIPI MORAIS OLIVEIRA,UGU FELIPI MORAIS OLIVEIRA,MARTINHA COSTA FERREIRA,INALDA MORAES OLIVEIRA,1998-05-18,1991-05-18
7,0.984569,sem_palavra,exato,diferente,0.626,PAULU AUVIS KEIROS FILU,PAULU AUVIS KEIROS FILU,JACIRA GOMES NUNES,ISABEL SANTANA QUEIROZ,1969-09-02,1979-09-02
8,0.984569,sem_palavra,exato,diferente,0.605,FELIPI PINHEIRU MONTEIRU,FELIPI PINHEIRU MONTEIRU,ROGELINA MONTEIRO MOURA,LUCELIA GOMES PINHEIRO,2007-07-23,2005-07-23
9,0.984569,sem_palavra,exato,diferente,0.656,RAIMUNDU AKINU PINHEIRU JUNIOR,RAIMUNDU AKINU PINHEIRU JUNIOR,LUCIMEIRE BARROS PINHEIRO,LUCILENE CONCEICAO MORAES,2021-12-10,2001-12-10


In [6]:
con.close()
